# Coach-change prediction

Compare a naive model, an SVM, and a simple PyTorch LSTM. Each model predicts `coach_changed` using the three previous seasons.

In [131]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

RANDOM_STATE = 42
TARGET = "coach_changed"
TEST_START_YEAR = 2005
HISTORY_YEARS = 3

teams = pd.read_csv(Path("data_prepared") / "teams.csv")
df = teams.sort_values(["franchID", "year"]).copy()

numeric_columns = [
    "rank", "G", "W", "L", "T", "OTL", "Pts", "SoW", "SoL", "GF", "GA",
    "PIM", "BenchMinor", "PPG", "PPC", "SHA", "PKG", "PKC", "SHF",
    "conference_rank", "prev_conference_rank", TARGET,
]
categorical_columns = []

for column in numeric_columns:
    for lag in range(1, HISTORY_YEARS + 1):
        df[f"previous_{lag}_{column}"] = df.groupby("franchID")[column].shift(lag)

sequence_columns = [
    column for column in numeric_columns
    if column != TARGET and df.loc[df["year"] <= TEST_START_YEAR - 4, column].notna().any()
]
previous_numeric = [
    f"previous_{lag}_{column}"
    for lag in range(1, HISTORY_YEARS + 1)
    for column in numeric_columns
    if df.loc[df["year"] <= TEST_START_YEAR - 4, f"previous_{lag}_{column}"].notna().any()
]
features = previous_numeric + categorical_columns
df = df[df["year"] >= df["year"].min() + HISTORY_YEARS].copy()
train = df[df["year"] < TEST_START_YEAR].copy()
test = df[df["year"] >= TEST_START_YEAR].copy()

def report_model(name, predictions):
    print(f"\n{name}")
    print(f"Accuracy:          {accuracy_score(test[TARGET], predictions):.2%}")
    print(f"Balanced accuracy: {balanced_accuracy_score(test[TARGET], predictions):.2%}")
    print(f"F1 (change=1):     {f1_score(test[TARGET], predictions, zero_division=0):.3f}")
    cm = confusion_matrix(test[TARGET], predictions)
    print(cm)
    cm_precision = np.divide(cm, cm.sum(axis=0, keepdims=True), out=np.zeros_like(cm, dtype=float), where=cm.sum(axis=0, keepdims=True) != 0)
    print(np.round(cm_precision, 3))

naive_predictions = np.zeros(len(test), dtype=int)
report_model("Naive model", naive_predictions)

numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, previous_numeric),
    ("categorical", categorical_pipeline, categorical_columns),
])



Naive model
Accuracy:          84.00%
Balanced accuracy: 50.00%
F1 (change=1):     0.000
[[126   0]
 [ 24   0]]
[[0.84 0.  ]
 [0.16 0.  ]]


In [ ]:
svm = Pipeline(
    [
        ("preprocessor", preprocessor),
        (
            "classifier",
            SVC(
                kernel="poly",
                degree=3,
                C=0.01,
                gamma=0.1,
                class_weight={0: 1, 1: 3},
                probability=True,
                random_state=RANDOM_STATE,
                tol=1e-3,
            ),
        ),
    ]
)

fit = train[train["year"] <= train["year"].max() - 3]
validation = train[train["year"] > train["year"].max() - 3]
svm.fit(fit[features], fit[TARGET])
validation_probability = svm.predict_proba(validation[features])[:, 1]
thresholds = np.arange(0.20, 0.81, 0.05)
threshold = float(
    thresholds[
        np.argmax(
            [
                f1_score(
                    validation[TARGET], validation_probability >= value, zero_division=0
                )
                for value in thresholds
            ]
        )
    ]
)
svm.fit(train[features], train[TARGET])
svm_predictions = (svm.predict_proba(test[features])[:, 1] >= threshold).astype(int)
report_model(f"SVM (threshold={threshold:.2f})", svm_predictions)


SVM (threshold=0.20)
Accuracy:          84.00%
Balanced accuracy: 50.00%
F1 (change=1):     0.000
[[126   0]
 [ 24   0]]
[[0.84 0.  ]
 [0.16 0.  ]]


C:\Users\barcl\AppData\Roaming\Python\Python311\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['previous_2_OTL']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\barcl\AppData\Roaming\Python\Python311\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['previous_2_OTL']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


In [138]:
from sklearn.neighbors import KNeighborsClassifier

knn = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", KNeighborsClassifier(
        n_neighbors=1,
        weights="distance",
        metric="euclidean",
    )),
])

# Train on the validation-training period
knn.fit(fit[features], fit[TARGET])

validation_probability = knn.predict_proba(
    validation[features]
)[:, 1]

# Select the threshold using validation data only
thresholds = np.arange(0.05, 0.81, 0.05)

validation_f1 = [
    f1_score(
        validation[TARGET],
        validation_probability >= threshold,
        zero_division=0,
    )
    for threshold in thresholds
]

knn_threshold = float(
    thresholds[int(np.argmax(validation_f1))]
)

print(f"Best validation F1: {max(validation_f1):.3f}")
print(f"Selected threshold: {knn_threshold:.2f}")

# Refit on all training seasons
knn.fit(train[features], train[TARGET])

test_probability = knn.predict_proba(
    test[features]
)[:, 1]

knn_predictions = (
    test_probability >= knn_threshold
).astype(int)

report_model(
    f"KNN (k=5, threshold={knn_threshold:.2f})",
    knn_predictions,
)

print(pd.crosstab(
    test[TARGET],
    knn_predictions,
    rownames=["Actual coach_changed"],
    colnames=["Predicted coach_changed"],
))

Best validation F1: 0.229
Selected threshold: 0.05

KNN (k=5, threshold=0.05)
Accuracy:          70.00%
Balanced accuracy: 53.47%
F1 (change=1):     0.237
[[98 28]
 [17  7]]
[[0.852 0.8  ]
 [0.148 0.2  ]]
Predicted coach_changed   0   1
Actual coach_changed           
0                        98  28
1                        17   7


C:\Users\barcl\AppData\Roaming\Python\Python311\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['previous_2_OTL']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\barcl\AppData\Roaming\Python\Python311\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['previous_2_OTL']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
